# LazImpa: Comparative Analysis
## LSTM vs Transformer with Lazy + Impatient Agents

This notebook analyzes the results of 4 experimental conditions:
1. **LSTM Baseline**: Standard agents with LSTM architecture
2. **LSTM LazImpa**: Lazy + Impatient agents with LSTM
3. **Transformer Baseline**: Standard agents with Transformer architecture
4. **Transformer LazImpa**: Lazy + Impatient agents with Transformer

## I - Setup and Load Data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 12

# Results directory
RESULTS_DIR = Path('results')

# Experiments to analyze (excluding test folders)
EXPERIMENTS = ['lstm_baseline', 'lstm_lazimpa', 'transformer_baseline', 'transformer_lazimpa']

# Colors for each experiment
COLORS = {
    'lstm_baseline': '#1f77b4',      # Blue
    'lstm_lazimpa': '#ff7f0e',        # Orange
    'transformer_baseline': '#2ca02c', # Green
    'transformer_lazimpa': '#d62728'   # Red
}

LABELS = {
    'lstm_baseline': 'LSTM Baseline',
    'lstm_lazimpa': 'LSTM LazImpa',
    'transformer_baseline': 'Transformer Baseline',
    'transformer_lazimpa': 'Transformer LazImpa'
}

print("Setup complete!")

In [ ]:
def load_messages(filepath):
    """Load messages from numpy file."""
    return np.load(filepath, allow_pickle=True)

def get_message_lengths(messages):
    """Calculate message lengths (position of EOS token which is 0)."""
    lengths = []
    for msg in messages:
        # EOS token is 0, find its first position
        eos_pos = np.where(msg == 0)[0]
        if len(eos_pos) > 0:
            lengths.append(eos_pos[0] + 1)  # Include EOS
        else:
            lengths.append(len(msg))
    return np.array(lengths)

def find_available_seeds(exp_dir):
    """Find all available seeds for an experiment."""
    seeds = []
    if exp_dir.exists():
        for seed_dir in exp_dir.glob('seed_*'):
            seed = int(seed_dir.name.split('_')[1])
            seeds.append(seed)
    return sorted(seeds)

# Check available data
print("Available data:")
print("="*60)
for exp in EXPERIMENTS:
    exp_dir = RESULTS_DIR / exp
    seeds = find_available_seeds(exp_dir)
    print(f"{LABELS[exp]}: {len(seeds)} seeds - {seeds}")

## II - Load All Experiment Data

In [ ]:
def load_experiment_data(exp_name):
    """Load all data for an experiment across all available seeds."""
    exp_dir = RESULTS_DIR / exp_name
    seeds = find_available_seeds(exp_dir)
    
    data = {'seeds': {}, 'name': exp_name}
    
    for seed in seeds:
        seed_dir = exp_dir / f'seed_{seed}'
        messages_dir = seed_dir / 'messages'
        accuracy_dir = seed_dir / 'accuracy'
        
        if not messages_dir.exists():
            continue
        
        # Find all epochs
        msg_files = sorted(messages_dir.glob('messages_*.npy'),
                          key=lambda x: int(x.stem.split('_')[1]))
        
        if not msg_files:
            continue
        
        # Load evolution data
        accuracies = []
        mean_lengths = []
        
        for mf in msg_files:
            epoch = int(mf.stem.split('_')[1])
            
            # Load messages and compute lengths
            messages = load_messages(mf)
            lengths = get_message_lengths(messages)
            mean_lengths.append(np.mean(lengths))
            
            # Load accuracy
            acc_file = accuracy_dir / f'accuracy_{epoch}.npy'
            if acc_file.exists():
                acc = np.load(acc_file)
                accuracies.append(np.mean(acc))
        
        # Load final epoch data
        final_messages = load_messages(msg_files[-1])
        final_lengths = get_message_lengths(final_messages)
        
        data['seeds'][seed] = {
            'accuracies': accuracies,
            'mean_lengths': mean_lengths,
            'final_messages': final_messages,
            'final_lengths': final_lengths,
            'n_epochs': len(msg_files)
        }
    
    return data

# Load all experiments
print("Loading experiment data...")
all_data = {}
for exp in EXPERIMENTS:
    print(f"  Loading {exp}...")
    all_data[exp] = load_experiment_data(exp)
    print(f"    -> {len(all_data[exp]['seeds'])} seeds loaded")

print("\nData loading complete!")

## III - Analyze Results

### III.1 - Message Length Distribution by Input Frequency

This plot shows how message length varies with input frequency (Zipf's Law of Abbreviation analysis).

In [ ]:
# Plot 1: Length distribution at different epochs (like original notebook cell 27)
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

epochs_to_show = [1, 50, 100, 199]  # Adjusted for 200 epochs
epoch_colors = ['#a6cee3', '#1f78b4', '#b2df8a', '#33a02c']

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx]
    exp_data = all_data[exp]
    
    if not exp_data['seeds']:
        ax.text(0.5, 0.5, 'No data available', ha='center', va='center', fontsize=14)
        ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
        continue
    
    # Use first available seed for epoch evolution
    seed = list(exp_data['seeds'].keys())[0]
    seed_dir = RESULTS_DIR / exp / f'seed_{seed}'
    
    for ep_idx, epoch in enumerate(epochs_to_show):
        msg_file = seed_dir / 'messages' / f'messages_{epoch}.npy'
        if msg_file.exists():
            messages = load_messages(msg_file)
            lengths = get_message_lengths(messages)
            ax.plot(lengths, label=f'Epoch {epoch}', color=epoch_colors[ep_idx], linewidth=2)
    
    ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
    ax.set_xlabel('Inputs ranked by frequency')
    ax.set_ylabel('Message length')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 32)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

plt.suptitle('Message Length as a Function of Input Frequency (Different Epochs)', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/analysis_length_by_epoch.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_length_by_epoch.png")

### III.2 - Accuracy Evolution

This plot shows how accuracy evolves during training for each experiment.

In [ ]:
# Plot 2: Accuracy evolution (like original notebook cell 29)
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

for exp in EXPERIMENTS:
    exp_data = all_data[exp]
    
    if not exp_data['seeds']:
        continue
    
    # Aggregate across seeds
    all_accs = [seed_data['accuracies'] for seed_data in exp_data['seeds'].values()]
    
    # Pad to same length
    max_len = max(len(acc) for acc in all_accs)
    padded = [acc + [acc[-1]] * (max_len - len(acc)) for acc in all_accs]
    
    mean_acc = np.mean(padded, axis=0)
    std_acc = np.std(padded, axis=0)
    
    epochs = range(len(mean_acc))
    ax.plot(epochs, mean_acc, label=LABELS[exp], color=COLORS[exp], linewidth=2.5)
    ax.fill_between(epochs, mean_acc - std_acc, mean_acc + std_acc, 
                    color=COLORS[exp], alpha=0.2)

ax.set_title('Accuracy Evolution During Training', fontsize=14, fontweight='bold')
ax.set_xlabel('Training Epochs')
ax.set_ylabel('Accuracy')
ax.set_xlim(0, 200)
ax.set_ylim(0, 1)
ax.legend(loc='lower right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/analysis_accuracy_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_accuracy_evolution.png")

### III.3 - Mean Message Length Evolution

This plot shows how the mean message length evolves during training.

In [ ]:
# Plot 3: Mean length evolution (like original notebook cell 31)
fig, ax = plt.subplots(1, 1, figsize=(12, 5))

for exp in EXPERIMENTS:
    exp_data = all_data[exp]
    
    if not exp_data['seeds']:
        continue
    
    # Aggregate across seeds
    all_lengths = [seed_data['mean_lengths'] for seed_data in exp_data['seeds'].values()]
    
    # Pad to same length
    max_len = max(len(l) for l in all_lengths)
    padded = [l + [l[-1]] * (max_len - len(l)) for l in all_lengths]
    
    mean_len = np.mean(padded, axis=0)
    std_len = np.std(padded, axis=0)
    
    epochs = range(len(mean_len))
    ax.plot(epochs, mean_len, label=LABELS[exp], color=COLORS[exp], linewidth=2.5)
    ax.fill_between(epochs, mean_len - std_len, mean_len + std_len,
                    color=COLORS[exp], alpha=0.2)

ax.set_title('Mean Message Length Evolution During Training', fontsize=14, fontweight='bold')
ax.set_xlabel('Training Epochs')
ax.set_ylabel('Mean Message Length')
ax.set_xlim(0, 200)
ax.set_ylim(0, 32)
ax.legend(loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('results/analysis_length_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_length_evolution.png")

### III.4 - Accuracy vs Mean Message Length

This plot shows the co-evolution of accuracy and mean message length, revealing when regularization (laziness) begins.

In [ ]:
# Plot 4: Accuracy vs Mean Length (like original notebook cell 33)
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx]
    exp_data = all_data[exp]
    
    if not exp_data['seeds']:
        ax.text(0.5, 0.5, 'No data available', ha='center', va='center', fontsize=14)
        ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
        continue
    
    for seed, seed_data in exp_data['seeds'].items():
        mean_lengths = seed_data['mean_lengths']
        accuracies = seed_data['accuracies']
        
        # Ensure same length
        min_len = min(len(mean_lengths), len(accuracies))
        
        ax.scatter(mean_lengths[:min_len], accuracies[:min_len], 
                  s=5, alpha=0.6, color=COLORS[exp], label=f'Seed {seed}')
    
    ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
    ax.set_xlabel('Mean Message Length')
    ax.set_ylabel('Accuracy')
    ax.set_xlim(0, 32)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)

plt.suptitle('Accuracy = f(Mean Message Length) - Training Evolution', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/analysis_accuracy_vs_length.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_accuracy_vs_length.png")

### III.5 - Final Message Length Distribution (Histogram)

In [ ]:
# Plot 5: Final length histograms
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx]
    exp_data = all_data[exp]
    
    if not exp_data['seeds']:
        ax.text(0.5, 0.5, 'No data available', ha='center', va='center', fontsize=14)
        ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
        continue
    
    # Aggregate final lengths across all seeds
    all_lengths = []
    for seed_data in exp_data['seeds'].values():
        all_lengths.extend(seed_data['final_lengths'])
    
    ax.hist(all_lengths, bins=range(1, 33), alpha=0.7, color=COLORS[exp], edgecolor='black')
    ax.axvline(np.mean(all_lengths), color='red', linestyle='--', linewidth=2,
              label=f'Mean: {np.mean(all_lengths):.2f}')
    
    ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
    ax.set_xlabel('Message Length')
    ax.set_ylabel('Frequency')
    ax.set_xlim(0, 32)
    ax.legend(loc='upper right')
    ax.grid(alpha=0.3)

plt.suptitle('Final Message Length Distribution', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/analysis_final_length_histogram.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_final_length_histogram.png")

### III.6 - Direct Comparison: LSTM vs Transformer

In [ ]:
# Plot 6: Direct comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Row 1: Baseline comparison (LSTM vs Transformer)
ax = axes[0, 0]
for exp in ['lstm_baseline', 'transformer_baseline']:
    exp_data = all_data[exp]
    if exp_data['seeds']:
        all_accs = [s['accuracies'] for s in exp_data['seeds'].values()]
        max_len = max(len(a) for a in all_accs)
        padded = [a + [a[-1]]*(max_len-len(a)) for a in all_accs]
        mean_acc = np.mean(padded, axis=0)
        ax.plot(mean_acc, label=LABELS[exp], color=COLORS[exp], linewidth=2.5)
ax.set_title('Baseline: Accuracy Evolution', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

ax = axes[0, 1]
for exp in ['lstm_baseline', 'transformer_baseline']:
    exp_data = all_data[exp]
    if exp_data['seeds']:
        all_lens = [s['mean_lengths'] for s in exp_data['seeds'].values()]
        max_len = max(len(l) for l in all_lens)
        padded = [l + [l[-1]]*(max_len-len(l)) for l in all_lens]
        mean_len = np.mean(padded, axis=0)
        ax.plot(mean_len, label=LABELS[exp], color=COLORS[exp], linewidth=2.5)
ax.set_title('Baseline: Length Evolution', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean Length')
ax.set_ylim(0, 32)
ax.legend()
ax.grid(alpha=0.3)

# Row 2: LazImpa comparison
ax = axes[1, 0]
for exp in ['lstm_lazimpa', 'transformer_lazimpa']:
    exp_data = all_data[exp]
    if exp_data['seeds']:
        all_accs = [s['accuracies'] for s in exp_data['seeds'].values()]
        max_len = max(len(a) for a in all_accs)
        padded = [a + [a[-1]]*(max_len-len(a)) for a in all_accs]
        mean_acc = np.mean(padded, axis=0)
        ax.plot(mean_acc, label=LABELS[exp], color=COLORS[exp], linewidth=2.5)
ax.set_title('LazImpa: Accuracy Evolution', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Accuracy')
ax.set_ylim(0, 1)
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1, 1]
for exp in ['lstm_lazimpa', 'transformer_lazimpa']:
    exp_data = all_data[exp]
    if exp_data['seeds']:
        all_lens = [s['mean_lengths'] for s in exp_data['seeds'].values()]
        max_len = max(len(l) for l in all_lens)
        padded = [l + [l[-1]]*(max_len-len(l)) for l in all_lens]
        mean_len = np.mean(padded, axis=0)
        ax.plot(mean_len, label=LABELS[exp], color=COLORS[exp], linewidth=2.5)
ax.set_title('LazImpa: Length Evolution', fontsize=12, fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Mean Length')
ax.set_ylim(0, 32)
ax.legend()
ax.grid(alpha=0.3)

plt.suptitle('Direct Comparison: LSTM vs Transformer', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/analysis_direct_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_direct_comparison.png")

### III.7 - Summary Statistics

In [ ]:
# Summary bar chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

final_metrics = {}
for exp in EXPERIMENTS:
    exp_data = all_data[exp]
    if exp_data['seeds']:
        final_accs = [s['accuracies'][-1] for s in exp_data['seeds'].values() if s['accuracies']]
        final_lens = []
        for s in exp_data['seeds'].values():
            final_lens.extend(s['final_lengths'])
        
        final_metrics[exp] = {
            'acc_mean': np.mean(final_accs),
            'acc_std': np.std(final_accs),
            'len_mean': np.mean(final_lens),
            'len_std': np.std(final_lens),
            'n_seeds': len(exp_data['seeds'])
        }

exps = list(final_metrics.keys())
x = range(len(exps))
colors = [COLORS[e] for e in exps]

# Accuracy bars
acc_means = [final_metrics[e]['acc_mean'] for e in exps]
acc_stds = [final_metrics[e]['acc_std'] for e in exps]
bars1 = ax1.bar(x, acc_means, yerr=acc_stds, capsize=5, color=colors)
ax1.set_ylabel('Final Accuracy', fontsize=12)
ax1.set_title('Final Accuracy Comparison', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels([LABELS[e].replace(' ', '\n') for e in exps], fontsize=10)
ax1.set_ylim(0, 1.1)
ax1.grid(alpha=0.3, axis='y')
for bar, mean in zip(bars1, acc_means):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{mean:.3f}', ha='center', va='bottom', fontsize=10)

# Length bars
len_means = [final_metrics[e]['len_mean'] for e in exps]
len_stds = [final_metrics[e]['len_std'] for e in exps]
bars2 = ax2.bar(x, len_means, yerr=len_stds, capsize=5, color=colors)
ax2.set_ylabel('Mean Message Length', fontsize=12)
ax2.set_title('Final Mean Message Length Comparison', fontsize=14, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels([LABELS[e].replace(' ', '\n') for e in exps], fontsize=10)
ax2.set_ylim(0, 35)
ax2.grid(alpha=0.3, axis='y')
for bar, mean in zip(bars2, len_means):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{mean:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('results/analysis_summary_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_summary_bars.png")

In [ ]:
# Print summary table
print("="*80)
print("EXPERIMENT RESULTS SUMMARY")
print("="*80)
print(f"{'Experiment':<25} {'Seeds':<8} {'Accuracy':<20} {'Mean Length':<20}")
print("-"*80)

for exp in EXPERIMENTS:
    if exp in final_metrics:
        m = final_metrics[exp]
        acc_str = f"{m['acc_mean']:.4f} +/- {m['acc_std']:.4f}"
        len_str = f"{m['len_mean']:.2f} +/- {m['len_std']:.2f}"
        print(f"{LABELS[exp]:<25} {m['n_seeds']:<8} {acc_str:<20} {len_str:<20}")

print("="*80)

# Key findings
print("\nKEY FINDINGS:")
print("-"*40)

if 'lstm_baseline' in final_metrics and 'lstm_lazimpa' in final_metrics:
    diff = final_metrics['lstm_baseline']['len_mean'] - final_metrics['lstm_lazimpa']['len_mean']
    print(f"LSTM: LazImpa reduces mean length by {diff:.2f} vs baseline")

if 'transformer_baseline' in final_metrics and 'transformer_lazimpa' in final_metrics:
    diff = final_metrics['transformer_baseline']['len_mean'] - final_metrics['transformer_lazimpa']['len_mean']
    print(f"Transformer: LazImpa reduces mean length by {diff:.2f} vs baseline")

if 'lstm_lazimpa' in final_metrics and 'transformer_lazimpa' in final_metrics:
    diff = final_metrics['lstm_lazimpa']['len_mean'] - final_metrics['transformer_lazimpa']['len_mean']
    if diff > 0:
        print(f"Transformer LazImpa achieves {diff:.2f} shorter messages than LSTM LazImpa")
    else:
        print(f"LSTM LazImpa achieves {-diff:.2f} shorter messages than Transformer LazImpa")

## IV - Zipf's Law of Abbreviation Analysis

Check if messages follow Zipf's Law of Abbreviation (shorter messages for more frequent inputs).

In [ ]:
# ZLA Analysis - Final epoch
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, exp in enumerate(EXPERIMENTS):
    ax = axes[idx]
    exp_data = all_data[exp]
    
    if not exp_data['seeds']:
        ax.text(0.5, 0.5, 'No data available', ha='center', va='center', fontsize=14)
        ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
        continue
    
    # Plot each seed
    for seed, seed_data in exp_data['seeds'].items():
        lengths = seed_data['final_lengths']
        ax.plot(lengths, alpha=0.5, color=COLORS[exp], linewidth=1.5)
    
    # Plot mean across seeds
    all_lens = [s['final_lengths'] for s in exp_data['seeds'].values()]
    min_n = min(len(l) for l in all_lens)
    truncated = [l[:min_n] for l in all_lens]
    mean_lens = np.mean(truncated, axis=0)
    ax.plot(mean_lens, color='black', linewidth=3, label='Mean')
    
    # Check ZLA compliance
    first_quarter = np.mean(mean_lens[:len(mean_lens)//4])
    last_quarter = np.mean(mean_lens[-len(mean_lens)//4:])
    zla_compliant = first_quarter < last_quarter
    
    compliance_text = "ZLA Compliant" if zla_compliant else "Anti-ZLA"
    compliance_color = 'green' if zla_compliant else 'red'
    ax.text(0.95, 0.95, compliance_text, transform=ax.transAxes, ha='right', va='top',
           fontsize=12, fontweight='bold', color=compliance_color,
           bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
    
    ax.set_title(LABELS[exp], fontsize=14, fontweight='bold')
    ax.set_xlabel('Input (ranked by frequency)')
    ax.set_ylabel('Message Length')
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 32)
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

plt.suptitle("Zipf's Law of Abbreviation Analysis (Final Epoch)", 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/analysis_zla_compliance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_zla_compliance.png")

## V - All Plots Combined

In [ ]:
# Create a combined figure with all key plots
fig = plt.figure(figsize=(18, 12))

# Subplot 1: Accuracy evolution
ax1 = fig.add_subplot(2, 3, 1)
for exp in EXPERIMENTS:
    exp_data = all_data[exp]
    if exp_data['seeds']:
        all_accs = [s['accuracies'] for s in exp_data['seeds'].values()]
        max_len = max(len(a) for a in all_accs)
        padded = [a + [a[-1]]*(max_len-len(a)) for a in all_accs]
        mean_acc = np.mean(padded, axis=0)
        ax1.plot(mean_acc, label=LABELS[exp], color=COLORS[exp], linewidth=2)
ax1.set_title('Accuracy Evolution', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0, 1)
ax1.legend(fontsize=8)
ax1.grid(alpha=0.3)

# Subplot 2: Length evolution
ax2 = fig.add_subplot(2, 3, 2)
for exp in EXPERIMENTS:
    exp_data = all_data[exp]
    if exp_data['seeds']:
        all_lens = [s['mean_lengths'] for s in exp_data['seeds'].values()]
        max_len = max(len(l) for l in all_lens)
        padded = [l + [l[-1]]*(max_len-len(l)) for l in all_lens]
        mean_len = np.mean(padded, axis=0)
        ax2.plot(mean_len, label=LABELS[exp], color=COLORS[exp], linewidth=2)
ax2.set_title('Mean Length Evolution', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Mean Length')
ax2.set_ylim(0, 32)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# Subplot 3: Final accuracy bars
ax3 = fig.add_subplot(2, 3, 3)
exps = [e for e in EXPERIMENTS if e in final_metrics]
x = range(len(exps))
acc_means = [final_metrics[e]['acc_mean'] for e in exps]
colors = [COLORS[e] for e in exps]
ax3.bar(x, acc_means, color=colors)
ax3.set_title('Final Accuracy', fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels([LABELS[e].replace(' ', '\n') for e in exps], fontsize=8)
ax3.set_ylim(0, 1.1)
ax3.grid(alpha=0.3, axis='y')

# Subplot 4: Final length bars
ax4 = fig.add_subplot(2, 3, 4)
len_means = [final_metrics[e]['len_mean'] for e in exps]
ax4.bar(x, len_means, color=colors)
ax4.set_title('Final Mean Length', fontweight='bold')
ax4.set_xticks(x)
ax4.set_xticklabels([LABELS[e].replace(' ', '\n') for e in exps], fontsize=8)
ax4.set_ylim(0, 35)
ax4.grid(alpha=0.3, axis='y')

# Subplot 5-6: ZLA plots for LazImpa variants
for plot_idx, exp in enumerate(['lstm_lazimpa', 'transformer_lazimpa']):
    ax = fig.add_subplot(2, 3, 5 + plot_idx)
    exp_data = all_data[exp]
    if exp_data['seeds']:
        all_lens = [s['final_lengths'] for s in exp_data['seeds'].values()]
        min_n = min(len(l) for l in all_lens)
        truncated = [l[:min_n] for l in all_lens]
        mean_lens = np.mean(truncated, axis=0)
        ax.plot(mean_lens, color=COLORS[exp], linewidth=2)
        ax.fill_between(range(len(mean_lens)), mean_lens, alpha=0.3, color=COLORS[exp])
    ax.set_title(f'{LABELS[exp]} - ZLA', fontweight='bold')
    ax.set_xlabel('Input (by frequency)')
    ax.set_ylabel('Message Length')
    ax.set_ylim(0, 32)
    ax.grid(alpha=0.3)

plt.suptitle('LazImpa: LSTM vs Transformer Comparative Analysis', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('results/analysis_combined_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/analysis_combined_summary.png")

In [ ]:
print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)
print("\nGenerated plots in results/ directory:")
print("  - analysis_length_by_epoch.png")
print("  - analysis_accuracy_evolution.png")
print("  - analysis_length_evolution.png")
print("  - analysis_accuracy_vs_length.png")
print("  - analysis_final_length_histogram.png")
print("  - analysis_direct_comparison.png")
print("  - analysis_summary_bars.png")
print("  - analysis_zla_compliance.png")
print("  - analysis_combined_summary.png")